In [18]:
from pathlib import Path
import pandas as pd
import duckdb
from pymongo import MongoClient

In [3]:
DATA_DIR = Path("data")

SALES_PATH = DATA_DIR / "processed" / "sales_train_evaluation.parquet"

OUTPUT_DIR = DATA_DIR / "processed" / "enriched_sales"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STORE_ID = "CA_1"
CHUNK_SIZE = 100

# Connect to MongoDB

In [4]:
client = MongoClient("mongodb://localhost:27017/")

db = client["m5_retail"]

calendar_collection = db["calendar"]
prices_collection = db["prices"]

print(client.admin.command("ping"))

{'ok': 1.0}


# Load sales Parquet

In [5]:
sales = pd.read_parquet(SALES_PATH)

print(sales.shape)
display(sales.head())

(30490, 1947)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [6]:
sales_store = sales[
    sales["store_id"] == STORE_ID
].copy()

print(sales_store.shape)

(3049, 1947)


# Load calendar from MongoDB

In [7]:
calendar_docs = list(
    calendar_collection.find(
        {},
        {"_id": 0}
    )
)

calendar = pd.DataFrame(calendar_docs)

calendar["date"] = pd.to_datetime(calendar["date"])

print(calendar.shape)
display(calendar.head())

(1969, 14)


,date,wm_yr_wk,weekday,wday,month,year,d,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2
0,2011-01-29,11101,Saturday,1,1,2011,d_1,0,0,0,NaN,NaN,NaN,NaN
1,2011-01-30,11101,Sunday,2,1,2011,d_2,0,0,0,NaN,NaN,NaN,NaN
2,2011-01-31,11101,Monday,3,1,2011,d_3,0,0,0,NaN,NaN,NaN,NaN
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,1,1,0,NaN,NaN,NaN,NaN
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,1,0,1,NaN,NaN,NaN,NaN


In [8]:
print(calendar["date"].min())
print(calendar["date"].max())
print(calendar["d"].nunique())

2011-01-29 00:00:00
2016-06-19 00:00:00
1969


# Load only CA_1 prices from MongoDB

In [9]:
price_docs = list(
    prices_collection.find(
        {"store_id": STORE_ID},
        {"_id": 0}
    )
)

prices_store = pd.DataFrame(price_docs)

print(prices_store.shape)
display(prices_store.head())

(698412, 4)


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [10]:
print("Items:", prices_store["item_id"].nunique())
print("Weeks:", prices_store["wm_yr_wk"].nunique())

print(
    "Duplicate price keys:",
    prices_store.duplicated(
        subset=["store_id", "item_id", "wm_yr_wk"]
    ).sum()
)

Items: 3049
Weeks: 282
Duplicate price keys: 0


# Identify daily sales columns

In [11]:
day_cols = [
    c for c in sales_store.columns
    if c.startswith("d_")
]

print("Days:", len(day_cols))
print(day_cols[:5])
print(day_cols[-5:])

Days: 1941
['d_1', 'd_2', 'd_3', 'd_4', 'd_5']
['d_1937', 'd_1938', 'd_1939', 'd_1940', 'd_1941']


# Build the enriched dataset in chunks

In [12]:
import math

n_chunks = math.ceil(len(day_cols) / CHUNK_SIZE)

print("Number of chunks:", n_chunks)

Number of chunks: 20


In [14]:
id_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

In [15]:
for chunk_num in range(n_chunks):

    start = chunk_num * CHUNK_SIZE
    end = start + CHUNK_SIZE

    current_days = day_cols[start:end]

    # Wide → long
    chunk = sales_store[
        id_cols + current_days
    ].melt(
        id_vars=id_cols,
        value_vars=current_days,
        var_name="d",
        value_name="units_sold"
    )

    # Join calendar
    chunk = chunk.merge(
        calendar,
        on="d",
        how="left",
        validate="many_to_one"
    )

    # Join weekly price
    chunk = chunk.merge(
        prices_store,
        on=[
            "store_id",
            "item_id",
            "wm_yr_wk"
        ],
        how="left",
        validate="many_to_one"
    )

    # Keep useful columns
    columns_to_keep = [
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "d",
        "date",
        "units_sold",
        "wm_yr_wk",
        "weekday",
        "wday",
        "month",
        "year",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
        "snap_CA",
        "snap_TX",
        "snap_WI",
        "sell_price"
    ]

    chunk = chunk[columns_to_keep]

    output_file = (
        OUTPUT_DIR /
        f"{STORE_ID}_part_{chunk_num:03d}.parquet"
    )

    chunk.to_parquet(
        output_file,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

    print(
        f"Saved chunk {chunk_num + 1}/{n_chunks}: "
        f"{len(chunk):,} rows"
    )

Saved chunk 1/20: 304,900 rows
Saved chunk 2/20: 304,900 rows
Saved chunk 3/20: 304,900 rows
Saved chunk 4/20: 304,900 rows
Saved chunk 5/20: 304,900 rows
Saved chunk 6/20: 304,900 rows
Saved chunk 7/20: 304,900 rows
Saved chunk 8/20: 304,900 rows
Saved chunk 9/20: 304,900 rows
Saved chunk 10/20: 304,900 rows
Saved chunk 11/20: 304,900 rows
Saved chunk 12/20: 304,900 rows
Saved chunk 13/20: 304,900 rows
Saved chunk 14/20: 304,900 rows
Saved chunk 15/20: 304,900 rows
Saved chunk 16/20: 304,900 rows
Saved chunk 17/20: 304,900 rows
Saved chunk 18/20: 304,900 rows
Saved chunk 19/20: 304,900 rows
Saved chunk 20/20: 125,009 rows


# Validate the output

In [16]:
files = sorted(
    OUTPUT_DIR.glob("CA_1_part_*.parquet")
)

print("Files:", len(files))

Files: 20


In [19]:
parquet_pattern = str(
    OUTPUT_DIR / "CA_1_part_*.parquet"
)

con = duckdb.connect()

In [20]:
con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{parquet_pattern}')
""").fetchone()

(5918109,)

In [21]:
con.execute(f"""
    SELECT
        MIN(date),
        MAX(date),
        COUNT(DISTINCT date)
    FROM read_parquet('{parquet_pattern}')
""").fetchdf()

,min(date),max(date),count(DISTINCT date)
0,2011-01-29,2016-05-22,1941


In [22]:
con.execute(f"""
    SELECT
        store_id,
        COUNT(*) AS rows
    FROM read_parquet('{parquet_pattern}')
    GROUP BY store_id
""").fetchdf()

,store_id,rows
0,CA_1,5918109


In [23]:
con.execute(f"""
    SELECT COUNT(DISTINCT item_id)
    FROM read_parquet('{parquet_pattern}')
""").fetchone()

(3049,)

# Check duplicate modeling keys

In [24]:
con.execute(f"""
    SELECT
        store_id,
        item_id,
        date,
        COUNT(*) AS n
    FROM read_parquet('{parquet_pattern}')
    GROUP BY
        store_id,
        item_id,
        date
    HAVING COUNT(*) > 1
    LIMIT 20
""").fetchdf()

,store_id,item_id,date,n


In [25]:
con.execute(f"""
    SELECT
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END)
            AS missing_dates,
        SUM(CASE WHEN wm_yr_wk IS NULL THEN 1 ELSE 0 END)
            AS missing_weeks
    FROM read_parquet('{parquet_pattern}')
""").fetchdf()

,missing_dates,missing_weeks
0,0.0,0.0


In [26]:
price_missing = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE WHEN sell_price IS NULL
            THEN 1 ELSE 0 END
        ) AS missing_price_rows,
        ROUND(
            100.0 *
            SUM(
                CASE WHEN sell_price IS NULL
                THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS missing_price_pct
    FROM read_parquet('{parquet_pattern}')
""").fetchdf()

price_missing

,total_rows,missing_price_rows,missing_price_pct
0,5918109,1129842.0,19.09


In [27]:
sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{parquet_pattern}')
    ORDER BY date, item_id
    LIMIT 20
""").fetchdf()

display(sample)

,item_id,dept_id,cat_id,store_id,state_id,d,date,units_sold,wm_yr_wk,weekday,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,3,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,2.00
1,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,0,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,7.88
2,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,0,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,2.88
3,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,0,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,NaN
4,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,3,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,2.94
5,FOODS_1_006,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,0,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,1.97
6,FOODS_1_008,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,0,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,NaN
7,FOODS_1_009,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,0,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,NaN
8,FOODS_1_010,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,0,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,NaN
9,FOODS_1_011,FOODS_1,FOODS,CA_1,CA,d_1,2011-01-29,2,11101,Saturday,...,1,2011,None,None,None,None,0,0,0,2.28


In [28]:
missing_price_analysis = con.execute(f"""
    WITH first_price AS (
        SELECT
            store_id,
            item_id,
            MIN(date) AS first_price_date
        FROM read_parquet('{parquet_pattern}')
        WHERE sell_price IS NOT NULL
        GROUP BY store_id, item_id
    )

    SELECT
        COUNT(*) AS missing_rows,
        SUM(
            CASE
                WHEN s.date < f.first_price_date THEN 1
                ELSE 0
            END
        ) AS before_first_price,
        SUM(
            CASE
                WHEN s.date >= f.first_price_date THEN 1
                ELSE 0
            END
        ) AS after_first_price
    FROM read_parquet('{parquet_pattern}') s
    LEFT JOIN first_price f
        ON s.store_id = f.store_id
        AND s.item_id = f.item_id
    WHERE s.sell_price IS NULL
""").fetchdf()

missing_price_analysis

,missing_rows,before_first_price,after_first_price
0,1129842,1129842.0,0.0


In [29]:
con.execute(f"""
    SELECT
        COUNT(*) AS missing_price_rows,
        SUM(CASE WHEN units_sold = 0 THEN 1 ELSE 0 END) AS zero_sales,
        SUM(CASE WHEN units_sold > 0 THEN 1 ELSE 0 END) AS positive_sales
    FROM read_parquet('{parquet_pattern}')
    WHERE sell_price IS NULL
""").fetchdf()

,missing_price_rows,zero_sales,positive_sales
0,1129842,1129842.0,0.0


In [30]:
stores = sales["store_id"].unique().tolist()
stores

['CA_1',
 'CA_2',
 'CA_3',
 'CA_4',
 'TX_1',
 'TX_2',
 'TX_3',
 'WI_1',
 'WI_2',
 'WI_3']

In [31]:
ALL_OUTPUT_DIR = DATA_DIR / "processed" / "enriched_sales_all"
ALL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for store_id in stores:

    print(f"\nProcessing {store_id}")

    sales_store = sales[
        sales["store_id"] == store_id
    ].copy()

    price_docs = list(
        prices_collection.find(
            {"store_id": store_id},
            {"_id": 0}
        )
    )

    prices_store = pd.DataFrame(price_docs)

    store_dir = ALL_OUTPUT_DIR / f"store_id={store_id}"
    store_dir.mkdir(parents=True, exist_ok=True)

    for chunk_num in range(n_chunks):

        start = chunk_num * CHUNK_SIZE
        end = start + CHUNK_SIZE
        current_days = day_cols[start:end]

        chunk = sales_store[
            id_cols + current_days
        ].melt(
            id_vars=id_cols,
            value_vars=current_days,
            var_name="d",
            value_name="units_sold"
        )

        chunk = chunk.merge(
            calendar,
            on="d",
            how="left",
            validate="many_to_one"
        )

        chunk = chunk.merge(
            prices_store,
            on=[
                "store_id",
                "item_id",
                "wm_yr_wk"
            ],
            how="left",
            validate="many_to_one"
        )

        # Availability feature
        chunk["is_available"] = (
            chunk["sell_price"]
            .notna()
            .astype("int8")
        )

        output_file = (
            store_dir /
            f"part_{chunk_num:03d}.parquet"
        )

        chunk.to_parquet(
            output_file,
            compression="snappy",
            index=False
        )

    print(f"Finished {store_id}")


Processing CA_1
Finished CA_1

Processing CA_2
Finished CA_2

Processing CA_3
Finished CA_3

Processing CA_4
Finished CA_4

Processing TX_1
Finished TX_1

Processing TX_2
Finished TX_2

Processing TX_3
Finished TX_3

Processing WI_1
Finished WI_1

Processing WI_2
Finished WI_2

Processing WI_3
Finished WI_3
